# Modelos Base

## 1. Librerias

In [23]:
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import s3fs
 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score


## 2. Configuración global

In [ ]:
S3_BASE = "s3://tfm-mu-bd/processed"
S3_OUT = "s3://tfm-mu-bd/outputs/baseline"
s3 = s3fs.S3FileSystem()

EXCLUDE_COLS = [
    "record_id", "dx", "labels", "categories",
    "is_adverse", "snomed_unknown", "dx_multi_hot", "cat_arrhythmia",
    "cat_axis_deviation", "cat_conduction_block", "cat_interval_change", "cat_ischemia",        
    "cat_morphology_change", "cat_repolarization",  
    "cat_sinus_rhythm", "cat_structural"   
]

imputer = SimpleImputer(strategy="median")

## 3. Carga de particiones y preparación

In [ ]:
def load_splits():
    print("Cargando datasets desde S3...")
    df_train = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_train.parquet")
    df_val = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_val.parquet")
    df_test = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_test.parquet")
 
    print(f" train: {df_train.shape} | val: {df_val.shape} | test: {df_test.shape}")
    return df_train, df_val, df_test
 
def prepare_xy(df, fit=False):
    feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
    X = df[feature_cols].values.astype(np.float32)
    y = df["is_adverse"].astype(int).values
    
    if fit:
        X = imputer.fit_transform(X)
    else:
        X = imputer.transform(X)
    
    return X, y, feature_cols

In [ ]:
def get_class_weights(y_train):
    classes = np.array([0, 1])
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    cw_dict = {0: weights[0], 1: weights[1]}
    print(f"class_weight: {cw_dict}")
    return cw_dict, weights[1] / weights[0]  

## 4. Entrenamiento y evaluación

In [ ]:
def evaluate(model, X, y, split_name, model_name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
 
    auc = roc_auc_score(y, y_prob)
    f1 = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)
 
    print(f"\n [{model_name}] {split_name}")
    print(f"AUC-ROC: {auc:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"Confusion matrix:\n{cm}")
 
    return {
        "model": model_name,
        "split": split_name,
        "auc_roc": round(auc,  4),
        "f1": round(f1,   4),
        "precision": round(prec, 4),
        "recall": round(rec,  4),
        "cm": cm.tolist()
    }

In [ ]:
def upload_artifacts(results_all):
    artifacts = [
        "./roc_curves.png",
        "./confusion_matrices.png",
        "./feature_importance.png",
        "./baseline_results.json",
    ]
    for local in artifacts:
        remote = f"{S3_OUT}/{local.lstrip('./')}"
        try:
            s3.put(local, remote)
            print(f" {local} a  {remote}")
        except Exception as e:
            print(f" ERROR subiendo {local}: {e}")


## 5. Ejecución 

In [ ]:
if __name__ == "__main__":
 
    df_train, df_val, df_test = load_splits()
    X_train, y_train, feature_cols = prepare_xy(df_train, fit=True)   
    X_val,   y_val,   _ = prepare_xy(df_val,   fit=False)   
    X_test,  y_test,  _= prepare_xy(df_test,  fit=False)   
    print(f"\nFeatures: {len(feature_cols)}")
    print(f"X_train shape: {X_train.shape}")
 
    print("\nCalculando class weights...")
    cw_dict, scale_pos_weight = get_class_weights(y_train)
 
    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
            n_jobs=-1
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            eval_metric="auc",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
    }
 
    results_all  = []
    results_test = []
    trained      = {}
 
    for name, model in models.items():
        print(f"\n{'='*50}")
        print(f"Entrenando: {name}")
        print(f"{'='*50}")
 
        if name == "XGBoost":
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        else:
            model.fit(X_train, y_train)
 
        trained[name] = model
 
        r_val  = evaluate(model, X_val,  y_val,  "val",  name)
        r_test = evaluate(model, X_test, y_test, "test", name)
 
        results_all.append(r_val)
        results_all.append(r_test)
        results_test.append(r_test)
 
        gc.collect()
 
    print(f"\n{'='*50}")
    print("RESUMEN COMPARATIVO (TEST)")
    print(f"{'='*50}")
    print(f"{'Modelo':<25} {'AUC-ROC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
    print("-" * 65)
    for r in results_test:
        print(f"{r['model']:<25} {r['auc_roc']:>8.4f} {r['f1']:>8.4f} "
              f"{r['precision']:>10.4f} {r['recall']:>8.4f}")
 
    print("\nGenerando visualizaciones...")
    plot_roc_curves(trained, X_test, y_test)
    plot_confusion_matrices(results_test)
    plot_feature_importance(trained["Random Forest"], feature_cols, top_n=20)
 
    with open("./baseline_results.json", "w") as f:
        json.dump(results_all, f, indent=2)
    print("  baseline_results.json guardado")
 
    print("\nSubiendo artefactos a S3...")
    upload_artifacts(results_all)
 
    print("\nBASELINE COMPLETO")
